# tinydistill: your own task

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andreas-Hum/tinydistill/blob/main/notebooks/your_task.ipynb)

Distil a classifier for **your** labels from **your** texts. You write the labels and a prompt; the texts need
no labels. As a stand-in for your data this notebook classifies news headlines by topic
([AG News](https://huggingface.co/datasets/fancyzhx/ag_news)); replace the task and the data with your own.

**Use a GPU runtime** (Runtime > Change runtime type > T4 GPU).

In [ ]:
!pip install -q "tinydistill[hf,plot]"

Downloads from the Hugging Face Hub work without an account. If they are slow, or you see a warning about
unauthenticated requests, add a token: in Colab, open the key icon in the left sidebar and add a secret named
`HF_TOKEN` with a token from https://huggingface.co/settings/tokens (read access is enough).

## 1. Write the task

- `prompt`: the instruction for the teacher, with `{labels}` (the list of options) and `{text}` (the input).
- `labels`: one entry per label. The name is what the teacher answers with, so keep it short and close to how
  the model would say it; a `description` is shown to the teacher in the list but not scored.
- `text_field` and `label_field`: the columns of your data that hold the text and, if you have them, gold
  labels. Gold labels are optional: without them you get the student but no accuracy numbers.

With gold labels, the label ids must match them: the gold labels' own names (as below, AG News's `World`,
`Sports`, `Business` and `Sci/Tech`), or the order of the labels when the gold labels are plain numbers. A
label's `name` is what the teacher answers with.

In [ ]:
%%writefile task.yaml
name: news-topics
text_field: text
label_field: label
prompt: |-
  Here is a news headline with the start of the article. What is it about?
  Options:
  {labels}

  News: {text}

  Reply with one option, exactly as written.
labels:  # the ids are the dataset's own label names; "name" is what the teacher answers with
  World:
    name: world
    description: politics, conflicts and events around the world
  Sports:
    name: sports
    description: games, athletes and competitions
  Business:
    name: business
    description: companies, markets and the economy
  Sci/Tech:
    name: science and technology
    description: research, computers, the internet and gadgets

## 2. Your data

Data can be a CSV or JSON Lines file with a text column, or a dataset on the Hugging Face Hub
(`hf:<dataset>:<split>`). In Colab, run the cell below to upload a CSV; otherwise the example data is used.
If your file has gold labels for some texts, keep a few hundred aside as the test set.

In [ ]:
DATA = "hf:fancyzhx/ag_news:train"
TEST = "hf:fancyzhx/ag_news:test"
TEACHER = "Qwen/Qwen2.5-1.5B-Instruct"

try:
    from google.colab import files

    if input("Upload your own CSV? [y/N] ").strip().lower() == "y":
        DATA = "/content/" + next(iter(files.upload()))
        TEST = ""  # no test set unless you upload one as well
except ImportError:
    pass  # not in Colab: use the example data
print("data:", DATA, "| test:", TEST or "(none)")

## 3. Check the teacher first

Small details of the task file can change the teacher a lot. Here, the label descriptions: with them,
`Qwen2.5-1.5B-Instruct` answers "world" for almost every AG News article; without them it gets most right.
Before labelling thousands of texts, `tune` scores a few hundred gold-labelled training texts with each
version of the task and ranks them, and the next step uses the winner. It needs gold labels; with your own
unlabelled data, the task file is used as it is.

In [ ]:
import json

import yaml

task = yaml.safe_load(open("task.yaml"))
labels = {i: v["name"] if isinstance(v, dict) else v for i, v in task["labels"].items()}
names_only = dict(task, labels=labels)
yaml.safe_dump(names_only, open("task-names-only.yaml", "w"), sort_keys=False)

TASK = "task.yaml"
if not DATA.startswith("/content/"):
    !tinydistill tune task.yaml --prompt task-names-only.yaml --data {DATA} --limit 3000 --dev-size 200 \
      --teacher {TEACHER} --out tune
    print(open("tune.md").read())
    TASK = json.load(open("tune.json"))[0]["prompt"]
print("using", TASK)

## 4. Distil

`--limit` keeps the first texts of the data; drop it to use everything.

In [ ]:
test_args = f"--test {TEST} --test-limit 500" if TEST else ""
!tinydistill run {TASK} --data {DATA} --limit 3000 {test_args} \
  --teacher {TEACHER} --val-share 0.05 --int8 --out run

In [ ]:
from IPython.display import Markdown

from tinydistill import plots

if TEST:
    display(Markdown(open("run/results.md").read()))
    plots.soft_labels("run")
    plots.per_label("run")
    plots.confusions("run")
plots.training("run");

## 5. Find the mistakes, label a few texts

`analyse` writes a page with the weakest labels and the most common confusions; a clearer label name or a
description usually helps more than anything on the student side. `suggest` picks the training texts most
worth labelling by hand; fill in their labels and retrain with `tinydistill train run --gold-file`.

In [ ]:
if TEST:
    !tinydistill analyse run
!tinydistill suggest run --n 20 --out to_label.jsonl
!head -n 5 to_label.jsonl

## 6. Take the model with you

The exported model is in `run/onnx` (about 23 MB as int8). Download it, then on any machine:
`pip install tinydistill` and `Predictor("onnx")`.

In [ ]:
!cd run && zip -qr ../model.zip onnx
try:
    from google.colab import files

    files.download("model.zip")
except ImportError:
    print("model.zip is ready")